# ContribGuide AI - Google Colab Setup

This notebook sets up the ContribGuide AI backend on Google Colab with GPU support.

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn pydantic pydantic-settings python-multipart pymupdf markdown sentence-transformers chromadb rank-bm25 google-generativeai httpx nest-asyncio pyngrok

In [ ]:
# Clone the repository (or upload files)
!git clone https://github.com/YOUR_USERNAME/contribguide-ai.git
%cd contribguide-ai/backend

In [ ]:
# Set up environment variables
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
os.environ["GEMMA_MODEL"] = "gemma-4-26b-a4b-it"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"
os.environ["CHROMA_PERSIST_DIRECTORY"] = "./data/chroma"
os.environ["UPLOAD_DIRECTORY"] = "./data/uploads"
os.environ["FRONTEND_ORIGIN"] = "*"

# Create directories
os.makedirs("./data/chroma", exist_ok=True)
os.makedirs("./data/uploads", exist_ok=True)

In [ ]:
# Test Google Generative AI connection
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# List available models
for m in genai.list_models():
    if "gemma" in m.name.lower():
        print(m.name)

In [ ]:
# Start the FastAPI server with ngrok tunnel
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn
import threading

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

# Run FastAPI in background thread
def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait a moment for server to start
import time
time.sleep(3)

# Test the API
import requests
response = requests.get(f"{public_url}/api/health")
print(response.json())

## Usage

1. Replace `YOUR_GOOGLE_API_KEY_HERE` with your actual API key
2. Replace `YOUR_USERNAME` with your GitHub username
3. Run all cells in order
4. The public URL from ngrok can be used as the API endpoint in your frontend

## Frontend Configuration

Update your frontend API constant to use the ngrok URL:
```javascript
const API = "https://your-ngrok-url.ngrok-free.app";
```

## Notes

- Colab sessions timeout after 90 minutes of inactivity
- Free tier has GPU limits (T4, K80)
- For production, consider Cloud Run, Vertex AI, or dedicated GPU instances
- The ngrok URL changes each session unless you have a paid plan